In [3]:
import os
import json
import pandas as pd
import traceback

  Using cached sqlalchemy-2.0.51-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
  Using cached greenlet-3.5.3-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 411.1 kB/s eta 0:00:05
   -------- ------------------------------- 0.5/2.4 MB 411.1 kB/s eta 0:00:05
   ------------- -------------------------- 0.8/2.4 MB 500.9 kB/s eta 0:00:04
   ------------- -------------------------- 0.8/2.4 MB 500.9 kB/s eta 0:00:04
   ----------------- ---------------------- 1.0/2.4 MB 539.7 kB/s eta 0:00:03
   ----------------- ---------------------- 1

In [165]:
if "HUGGINGFACEHUB_API_TOKEN" in os.environ:
    del os.environ["HUGGINGFACEHUB_API_TOKEN"]

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
pip install langchain_huggingface

   ---------------------------------------- 0.0/765.1 kB ? eta -:--:--
   ------------- -------------------------- 262.1/765.1 kB ? eta -:--:--
   -------------------------- ----------- 524.3/765.1 kB 982.2 kB/s eta 0:00:01
   ---------------------------------------- 765.1/765.1 kB 1.2 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 978.6 kB/s eta 0:00:04
   ------- -------------------------------- 0.8/4.0 MB 973.3 kB/s eta 0:00:04
   ---------- ----------------------------- 1.0/4.0 MB 969.7 kB/s eta 0:00:04
   ---------- ----------------------------- 1.0/4.0 MB 969.7 kB/s eta 0:00:04
   ------------- -------------------------- 1.3/4.0 MB 971.5 kB/s eta 0:00:03
   --------------- ------------------------ 1.6/4.0 MB 971.7 kB/s eta 0:00:03
   ------------------ --

In [7]:
import os
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Setup your endpoint and chat model (as you already did)
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct", # Cleaned up repo id if using standard HF
    task="text-generation",
    temperature=0.5,
    max_new_tokens=512
)
chat = ChatHuggingFace(llm=llm)

# 2. Define an output parser to extract clean string text from the model responses
output_parser = StrOutputParser()

# ----------------------------------------------------
# CHAIN 1: Generate a company name based on a product
# ----------------------------------------------------
prompt1 = ChatPromptTemplate.from_template(
    "What is a good, catchy name for a company that makes {product}?"
)
# Construct the first chain
chain_one = prompt1 | chat | output_parser

# ----------------------------------------------------
# CHAIN 2: Generate a tagline based on the company name
# ----------------------------------------------------
prompt2 = ChatPromptTemplate.from_template(
    "Write a catchy 3-word slogan or tagline for the following company: {company_name}"
)
# Construct the second chain
chain_two = prompt2 | chat | output_parser

print(output_parser)

# ----------------------------------------------------
# THE SEQUENTIAL CHAIN: Link them together
# ----------------------------------------------------
# We use a lambda function to map the output of chain_one into the input key of chain_two
sequential_chain = chain_one | (lambda company: {"company_name": company}) | chain_two

# 4. Run the complete pipeline
result = sequential_chain.invoke({"product": "eco-friendly running shoes"})
print(result)


e:\Personal Project\First-Ai-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Here are three catchy 3-word slogans/taglines that align with your eco-friendly running shoe brand:

1. **"Run Light. Run Green."** *(Simple, minimalist, and eco-focused)*
2. **"Step Forward Sustainably."** *(Balances performance and sustainability)*
3. **"Nature’s Pace, Zero Waste."** *(Strong eco-message with a running twist)*

If you'd like a more **performance-driven** tagline, try:
**"Fast. Fresh. Footprint-Free."**

Or a **minimalist** one:
**"Tread Light. Go Far."**

Would you like me to refine further based on a specific vibe? 😊


In [102]:
import os
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
# 1. Setup your endpoint and chat model (as you already did)
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct", # Cleaned up repo id if using standard HF
    task="text-generation",
    temperature=0.5,
    max_new_tokens=512
)
chat = ChatHuggingFace(llm=llm)

In [103]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [111]:
TEMPLATE="""
Text{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sute the questions are not repeated and check all the questions to be confirmed the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. Strictly follow the format \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""
chain_one = ChatPromptTemplate.from_template(TEMPLATE) | chat | output_parser

In [112]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""
chain_two = ChatPromptTemplate.from_template(TEMPLATE2) | chat | output_parser

In [113]:
from langchain_core.runnables import RunnablePassthrough

# Assuming 'quiz_chain' and 'review_chain' are already defined earlier in your code
# (e.g., quiz_chain = prompt1 | chat | output_parser)

generate_evaluate_chain = (
    # 1. Run quiz_chain and append its output to the running state dictionary as 'quiz'
    RunnablePassthrough.assign(quiz=chain_one)
    
    # 2. Run review_chain using the updated state (which now includes 'quiz' and 'subject') 
    #    and append its output as 'review'
    | RunnablePassthrough.assign(review=chain_two)
)

In [107]:
file_path = r'E:\Personal Project\First-Ai-Project\data.txt'

with open(file_path,'r') as file:
    TEXT = file.read()

# print(TEXT)

In [108]:
# Conveting the python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [109]:
NUMBER = 5
SUBJECT ='globalization in india'
TONE = 'simple'

In [114]:
from langchain_community.callbacks import get_openai_callback

# Wrap your invoke call inside the context manager
with get_openai_callback() as cb:
    result = generate_evaluate_chain.invoke({
        'text': TEXT,
        'number': NUMBER,
        'subject': SUBJECT,
        'tone': TONE,
        'response_json': json.dumps(RESPONSE_JSON)
    })
    
    # Print the token breakdown
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")

# Your existing code continues down here
text_output = result.get('quiz') if isinstance(result, dict) else result
print(text_output)
quiz = parser.parse(text_output)

Total Tokens: 4812
Prompt Tokens: 3849
Completion Tokens: 963
Total Cost (USD): $0.0
```json
{
  "1": {
    "mcq": "What percentage of the world's GDP did India account for at the end of the Mughal era?",
    "options": {
      "a": "10.5%",
      "b": "17%",
      "c": "25.3%",
      "d": "32.9%"
    },
    "correct": "d"
  },
  "2": {
    "mcq": "Which event significantly boosted India's technology sector and job opportunities?",
    "options": {
      "a": "The launch of the first Indian satellite",
      "b": "Netscape's IPO in 1995",
      "c": "The establishment of the World Trade Organization",
      "d": "The 2008 financial crisis"
    },
    "correct": "b"
  },
  "3": {
    "mcq": "What was the primary reason for India's economic isolation before 1991?",
    "options": {
      "a": "Lack of natural resources",
      "b": "To protect its fledgling economy and achieve self-reliance",
      "c": "High levels of foreign investment",
      "d": "Political instability"
    },
    "c

In [115]:
print(quiz)

{'1': {'mcq': "What percentage of the world's GDP did India account for at the end of the Mughal era?", 'options': {'a': '10.5%', 'b': '17%', 'c': '25.3%', 'd': '32.9%'}, 'correct': 'd'}, '2': {'mcq': "Which event significantly boosted India's technology sector and job opportunities?", 'options': {'a': 'The launch of the first Indian satellite', 'b': "Netscape's IPO in 1995", 'c': 'The establishment of the World Trade Organization', 'd': 'The 2008 financial crisis'}, 'correct': 'b'}, '3': {'mcq': "What was the primary reason for India's economic isolation before 1991?", 'options': {'a': 'Lack of natural resources', 'b': 'To protect its fledgling economy and achieve self-reliance', 'c': 'High levels of foreign investment', 'd': 'Political instability'}, 'correct': 'b'}, '4': {'mcq': 'Which sector in India has seen a decline in growth due to globalization and reduced funding?', 'options': {'a': 'Technology', 'b': 'Agriculture', 'c': 'Manufacturing', 'd': 'Healthcare'}, 'correct': 'b'}, '

In [116]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})
    print(quiz_table_data)

[{'MCQ': "What percentage of the world's GDP did India account for at the end of the Mughal era?", 'Choices': 'a: 10.5% | b: 17% | c: 25.3% | d: 32.9%', 'Correct': 'd'}]
[{'MCQ': "What percentage of the world's GDP did India account for at the end of the Mughal era?", 'Choices': 'a: 10.5% | b: 17% | c: 25.3% | d: 32.9%', 'Correct': 'd'}, {'MCQ': "Which event significantly boosted India's technology sector and job opportunities?", 'Choices': "a: The launch of the first Indian satellite | b: Netscape's IPO in 1995 | c: The establishment of the World Trade Organization | d: The 2008 financial crisis", 'Correct': 'b'}]
[{'MCQ': "What percentage of the world's GDP did India account for at the end of the Mughal era?", 'Choices': 'a: 10.5% | b: 17% | c: 25.3% | d: 32.9%', 'Correct': 'd'}, {'MCQ': "Which event significantly boosted India's technology sector and job opportunities?", 'Choices': "a: The launch of the first Indian satellite | b: Netscape's IPO in 1995 | c: The establishment of the

In [117]:
pd.DataFrame(quiz_table_data)

,MCQ,Choices,Correct
0,What percentage of the world's GDP did India a...,a: 10.5% | b: 17% | c: 25.3% | d: 32.9%,d
1,Which event significantly boosted India's tech...,a: The launch of the first Indian satellite | ...,b
2,What was the primary reason for India's econom...,a: Lack of natural resources | b: To protect i...,b
3,Which sector in India has seen a decline in gr...,a: Technology | b: Agriculture | c: Manufactur...,b
4,What is one of the main factors that has contr...,a: Increased agricultural subsidies | b: The h...,b


In [101]:
from datetime import datetime
datetime.now().strftime('%d/%m/%Y, %H:%M:%S')

'06/07/2026, 14:46:07'